### G1 — Pertanyaan analisis

**Dataset proyek:** `French bakery daily sales` (Kaggle)

**Pertanyaan payung:**
> Bagaimana pola penjualan produk bakery berdasarkan jenis produk, waktu transaksi, dan harga?

| No. | Pertanyaan analisis | Kolom pendukung | Uji kolom | Uji bentuk jawaban | Uji tindak lanjut |
|---|---|---|---|---|---|
| 1 | Produk apa yang memiliki jumlah penjualan paling tinggi? | `article`, `Quantity` | Lolos | Lolos — peringkat produk berdasarkan jumlah penjualan | Lolos — membantu melihat produk dengan volume penjualan tertinggi |
| 2 | Bagaimana perubahan jumlah penjualan berdasarkan tanggal? | `date`, `Quantity` | Lolos | Lolos — angka/sebaran penjualan dari waktu ke waktu | Lolos — membantu melihat pola penjualan harian |
| 3 | Pada waktu kapan transaksi paling banyak terjadi? | `time`, `ticket_number` | Lolos | Lolos — peringkat/sebaran jumlah transaksi menurut waktu | Lolos — membantu melihat waktu transaksi yang paling ramai |
| 4 | Bagaimana hubungan harga satuan dengan jumlah produk yang terjual? | `unit_price`, `Quantity` | Lolos | Lolos — perbandingan/hubungan dua besaran | Lolos — membantu melihat pola penjualan menurut harga |

**Hasil uji yang sudah dijalankan:**
- Pertanyaan 1 sudah diuji dengan `groupby('article')['Quantity'].sum()` dan menghasilkan **TRADITIONAL BAGUETTE = 117.463 unit** sebagai hasil eksplorasi awal.
- Nilai tersebut masih **hasil sebelum tahap cleaning**, karena ditemukan `Quantity` negatif pada 1.295 baris.
- Pertanyaan 2–4 sudah dinilai berdasarkan ketersediaan kolom dan bentuk jawaban, tetapi perhitungan hasil datanya belum dijalankan pada tahap ini.


### G2 — Provenans data

| Butir | Isi |
|---|---|
| Nama sumber | **French bakery daily sales** |
| Tautan atau nama instansi | **Kaggle — dataset French bakery daily sales**: https://www.kaggle.com/datasets/matthieugimbert/french-bakery-daily-sales/data |
| Tanggal pengambilan | **5 Oktober 2026** |
| Lisensi atau nomor surat izin | **Data files © Original Authors** |
| Kalimat sitasi untuk laporan | **Sumber data: French bakery daily sales, Kaggle. https://www.kaggle.com/datasets/matthieugimbert/french-bakery-daily-sales/data** |

> Catatan: tanggal pengambilan sebaiknya disesuaikan dengan tanggal saat file benar-benar diunduh jika berbeda dari tanggal pengerjaan notebook.


### G3 — Kamus data

Kamus data berikut disusun untuk seluruh kolom pada file proyek. Kolom `Unnamed: 0` merupakan indeks tambahan pada file CSV dan tidak diperlukan untuk pertanyaan analisis, sehingga tindakannya **hapus**.

| Nama kolom | Deskripsi | Tipe data / skala | Satuan | Nilai kosong | Sumber | Kategori data pribadi | Tindakan penanganan |
|---|---|---|---|---:|---|---|---|
| `date` | Tanggal ketika baris transaksi tercatat. | datetime / interval | tanggal | 0 | French bakery daily sales | bukan data pribadi | pertahankan |
| `time` | Waktu ketika transaksi tercatat pada hari tersebut. | time / interval | waktu | 0 | French bakery daily sales | bukan data pribadi | pertahankan |
| `ticket_number` | Nomor tiket yang menandai satu transaksi penjualan. | integer / nominal | ID transaksi | 0 | French bakery daily sales | bukan data pribadi | pertahankan |
| `article` | Nama produk bakery yang tercatat pada transaksi. | string / nominal | nama produk | 0 | French bakery daily sales | bukan data pribadi | pertahankan |
| `Quantity` | Jumlah unit produk yang tercatat pada baris transaksi. | integer / rasio | unit | 0 | French bakery daily sales | bukan data pribadi | pertahankan sementara; nilai negatif diperiksa pada tahap cleaning |
| `unit_price` | Harga satuan produk pada transaksi. | float / rasio | euro (€) | 0 | French bakery daily sales | bukan data pribadi | pertahankan sementara; nilai 0 diperiksa pada tahap cleaning |
| `Unnamed: 0` | Indeks/baris tambahan yang ikut tersimpan ketika CSV dibuat dan tidak memiliki makna analisis. | integer / nominal | indeks | 0 | File CSV proyek | bukan data pribadi | **hapus** |

**Penandaan data pribadi:** tidak ditemukan pengenal langsung, quasi-identifier, atau data spesifik pada tujuh kolom dataset ini. Karena tidak ada quasi-identifier, **k-anonymity tidak berlaku** untuk dataset proyek ini.


In [ ]:
# G4 — jalankan penanganan pada data tim dan periksa nilai k
# Dataset proyek: French bakery daily sales
# Catatan: penanganan di bagian ini berfokus pada metadata/data pribadi.
# Nilai Quantity negatif dan unit_price = 0 dicatat untuk pemeriksaan kualitas
# pada tahap cleaning berikutnya, bukan langsung dihapus pada M1.

import os
import pandas as pd

FILE = 'Bakery sales.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        print("Silakan unggah berkas 'Bakery sales.csv' terlebih dahulu:")
        files.upload()          # Pilih file Bakery sales.csv
    except ImportError:
        raise FileNotFoundError(
            f"File '{FILE}' tidak ditemukan. Letakkan '{FILE}' di folder yang sama dengan notebook ini."
        )

# 1. Muat data
data_tim = pd.read_csv(FILE)

# 2. Konversi tipe data yang diperlukan
data_tim['date'] = pd.to_datetime(data_tim['date'], errors='coerce')
data_tim['time'] = pd.to_datetime(
    data_tim['time'].astype(str), format='%H:%M:%S', errors='coerce'
).dt.time
data_tim['unit_price'] = (
    data_tim['unit_price']
    .astype(str)
    .str.replace('€', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .astype(float)
)

# 3. Penanganan kolom yang tidak diperlukan untuk analisis
kolom_dihapus = ['Unnamed: 0']
data_processed = data_tim.drop(columns=kolom_dihapus, errors='ignore').copy()

# 4. Periksa temuan kualitas yang akan dibawa ke tahap cleaning
jumlah_quantity_negatif = int((data_processed['Quantity'] < 0).sum())
jumlah_harga_nol = int((data_processed['unit_price'] == 0).sum())

print('Jumlah baris setelah penanganan:', len(data_processed))
print('Kolom yang dipertahankan:', data_processed.columns.tolist())
print('Quantity negatif (dicatat untuk cleaning):', jumlah_quantity_negatif)
print('unit_price = 0 (dicatat untuk cleaning):', jumlah_harga_nol)

# 5. Periksa k-anonymity berdasarkan quasi-identifier dari kamus data
quasi_identifier = []

if len(quasi_identifier) == 0:
    nilai_k = 'Tidak berlaku'
    keterangan_k = (
        'Tidak terdapat quasi-identifier atau data pribadi pada dataset, '
        'sehingga k-anonymity tidak perlu diterapkan.'
    )
else:
    grup = data_processed.groupby(quasi_identifier).size()
    nilai_k = int(grup.min())
    keterangan_k = 'Nilai k dihitung dari ukuran grup terkecil berdasarkan quasi-identifier.'

print('Quasi-identifier:', quasi_identifier)
print('Nilai k:', nilai_k)
print('Keterangan:', keterangan_k)

# 6. Simpan versi terproses untuk analisis, tanpa kolom indeks tambahan
os.makedirs('data/processed', exist_ok=True)
output_file = 'data/processed/Bakery_sales_processed.csv'
data_processed.to_csv(output_file, index=False)
print('File hasil penanganan:', output_file)

Silakan unggah berkas 'Bakery sales.csv' terlebih dahulu:


Saving Bakery sales.csv to Bakery sales.csv
Jumlah baris setelah penanganan: 234005
Kolom yang dipertahankan: ['date', 'time', 'ticket_number', 'article', 'Quantity', 'unit_price']
Quantity negatif (dicatat untuk cleaning): 1295
unit_price = 0 (dicatat untuk cleaning): 32
Quasi-identifier: []
Nilai k: Tidak berlaku
Keterangan: Tidak terdapat quasi-identifier atau data pribadi pada dataset, sehingga k-anonymity tidak perlu diterapkan.
File hasil penanganan: data/processed/Bakery_sales_processed.csv


### G5 — Daftar periksa sebelum `git push`

Status berikut khusus untuk proyek **French bakery daily sales** berdasarkan pengerjaan M1 kita:

- [ ] `data/raw/` masuk `.gitignore` — **perlu dicek di repositori**.
- [x] Tidak ada nama, NIK, nomor telepon, surel, atau alamat pada kolom dataset yang dianalisis.
- [ ] Keluaran sel notebook yang sempat menampilkan data pribadi sudah dibersihkan — **cek seluruh notebook sebelum commit**.
- [x] Garam hash tidak digunakan pada proyek bakery karena tidak ada kolom identitas yang perlu di-hash.
- [x] Kamus data sudah memuat deskripsi, skala, kategori data pribadi, dan tindakan untuk setiap kolom.
- [x] Nilai k sudah diperiksa dan dicatat sebagai **Tidak berlaku** karena tidak terdapat quasi-identifier/data pribadi pada dataset.
- [ ] Bila masih ragu: repositori dibuat **privat**.

> Temuan kualitas data yang **belum dibersihkan** pada tahap M1: `Quantity` memiliki 1.295 nilai negatif dan `unit_price` memiliki 32 nilai nol. Temuan ini dibawa ke tahap pemeriksaan/kualitas data berikutnya.

> Menghapus berkas lalu *commit* ulang **tidak** menghilangkan data dari riwayat Git. Periksa **sebelum** *commit* pertama.
